# AttnRes VLM Ablation (Colab / CUDA)

Exploratory Full/Block Attention Residuals study on a tiny modular VLM.

Open with a GPU runtime and **Run all**. Persistent artifacts go under `MyDrive/SharedColab/attnres_vlm_ablation`. W&B project: `attnres-next-scale-vlm`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%cd /content
!rm -rf /content/AttnResGPT-next-scale
!git clone https://github.com/AtinChing/AttnResGPT-next-scale.git /content/AttnResGPT-next-scale
%cd /content/AttnResGPT-next-scale
!pip install -q -r requirements.txt


In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/AttnResGPT-next-scale")
DRIVE_ROOT = Path("/content/drive/MyDrive")
SHARED_COLAB_ROOT = DRIVE_ROOT / "SharedColab"
PROJECT_ROOT = SHARED_COLAB_ROOT / "attnres_vlm_ablation"

SHARED_COLAB_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for name in [
    "source",
    "configs",
    "checkpoints",
    "runs",
    "logs",
    "metrics",
    "summaries",
    "plots",
    "tables",
    "cache",
    "manifests",
]:
    (PROJECT_ROOT / name).mkdir(parents=True, exist_ok=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("REPO:", REPO)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


## Weights & Biases login


In [ ]:
import os
import wandb

# Prefer env var / Colab secret. Do not hardcode API keys in the notebook.
try:
    from google.colab import userdata

    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale-vlm"
WANDB_ENABLED = True
WANDB_MODE = "auto"  # auto | online | offline | disabled
WANDB_RESUME = "allow"  # allow | must | never

wandb.login()
print("W&B project:", WANDB_PROJECT)
print("W&B entity:", WANDB_ENTITY)


## Configuration


In [ ]:
RUN_MODE = "quick"  # "smoke", "quick", or "full"

RESUME = True
FORCE_RESTART = False

RUN_PRIMARY_FULL_GRID = True
RUN_BLOCK_GRID = True

SEEDS = [0, 1, 2]

PRIMARY_VARIANTS = [
    "baseline",
    "encoder_full",
    "decoder_full",
    "both_full",
]

BLOCK_VARIANTS = [
    "encoder_block",
    "decoder_block",
    "both_block",
]

# Training / model knobs
BATCH_SIZE = 64
NUM_WORKERS = 2
CHECKPOINT_INTERVAL = 100
EVALUATION_INTERVAL = 1
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
MIXED_PRECISION = True
AMP_DTYPE = "auto"  # "auto" | "float16" | "bfloat16"

VISION_D_MODEL = 128
VISION_N_LAYERS = 6
VISION_N_HEADS = 4
VISION_D_FF = 512
DECODER_D_MODEL = 128
DECODER_N_LAYERS = 6
DECODER_N_HEADS = 4
DECODER_D_FF = 512
NUM_BLOCKS = 3
DROPOUT = 0.0

print("RUN_MODE:", RUN_MODE)
print("PRIMARY_VARIANTS:", PRIMARY_VARIANTS)
print("BLOCK_VARIANTS:", BLOCK_VARIANTS)
print("SEEDS:", SEEDS)
print("WANDB_PROJECT:", WANDB_PROJECT)


## CUDA enforcement


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Select a Colab GPU runtime and rerun. "
        "Training will not fall back to CPU or MPS."
    )

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
major, minor = torch.cuda.get_device_capability(0)
selected_amp = "bfloat16" if (AMP_DTYPE == "auto" and major >= 8) else (
    "float16" if AMP_DTYPE == "auto" else AMP_DTYPE
)

print("GPU:", props.name)
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("Total GPU memory bytes:", int(total_bytes))
print("Available GPU memory bytes:", int(free_bytes))
print("Selected AMP dtype:", selected_amp)


## Run experiment


In [ ]:
from src.vlm.ablation.config import AblationExperimentConfig
from src.vlm.ablation.runner import run_ablation_experiment

config = AblationExperimentConfig(
    run_mode=RUN_MODE,
    resume=RESUME,
    force_restart=FORCE_RESTART,
    run_primary_full_grid=RUN_PRIMARY_FULL_GRID,
    run_block_grid=RUN_BLOCK_GRID,
    seeds=list(SEEDS),
    primary_variants=list(PRIMARY_VARIANTS),
    block_variants=list(BLOCK_VARIANTS),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    evaluation_interval=EVALUATION_INTERVAL,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=MIXED_PRECISION,
    amp_dtype=AMP_DTYPE,
    vision_d_model=VISION_D_MODEL,
    vision_n_layers=VISION_N_LAYERS,
    vision_n_heads=VISION_N_HEADS,
    vision_d_ff=VISION_D_FF,
    decoder_d_model=DECODER_D_MODEL,
    decoder_n_layers=DECODER_N_LAYERS,
    decoder_n_heads=DECODER_N_HEADS,
    decoder_d_ff=DECODER_D_FF,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    project_root=str(PROJECT_ROOT),
    wandb_enabled=WANDB_ENABLED,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_mode=WANDB_MODE,
    wandb_resume=WANDB_RESUME,
)

summary = run_ablation_experiment(
    config,
    project_root=PROJECT_ROOT,
    source_root=REPO,
)
summary


## Completion summary


In [ ]:
import json

summary_path = PROJECT_ROOT / "summaries" / "experiment_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Mounted Drive root:", DRIVE_ROOT)
print("Experiment root:", PROJECT_ROOT)
print("Repo clone:", REPO)
print("GPU information:", json.dumps(summary.get("cuda", {}), indent=2))
print("Run mode:", summary.get("run_mode"))
print("W&B project:", summary.get("wandb_project"))
print("W&B entity:", summary.get("wandb_entity"))
print("W&B summary run:", summary.get("wandb_summary"))
print("Requested variants:", summary.get("requested_variants"))
print("Completed:", summary.get("completed"))
print("Resumed:", summary.get("resumed"))
print("Failed:", summary.get("failed"))
print("Best by variant:")
for variant, metrics in (summary.get("best_by_variant") or {}).items():
    print(
        f"  {variant}: val={metrics.get('validation_accuracy')} "
        f"test={metrics.get('test_accuracy')} "
        f"best_ckpt={metrics.get('checkpoint_best')}"
    )
print("Aggregate CSV paths:", summary.get("tables"))
print("Plot directory:", summary.get("plots_dir"))
print("All persistent artifacts under Google Drive:", summary.get("artifacts_under_drive"))
assert str(PROJECT_ROOT).startswith(str(DRIVE_ROOT)), "Project root escaped Drive"
print("Idempotent re-run safe: completed runs are skipped via completed.marker")
